# AEI Index Audit — robustness pack

Five checks, each attacking a specific referee question. Run after
`01_full_analysis.ipynb`. About 3 minutes.

**Nothing here is decoration.** A robustness check that cannot change a
conclusion should be deleted, not run.

In [ ]:
!pip install -q numpy pandas matplotlib plotly
import os, sys, subprocess
from google.colab import userdata
GH_USER, REPO = "vkenned2", "aei-index-audit"
try: token = userdata.get("GH_TOKEN")
except Exception: token = None
if not os.path.exists(REPO):
    url = f"https://{token}@github.com/{GH_USER}/{REPO}.git" if token else f"https://github.com/{GH_USER}/{REPO}.git"
    subprocess.run(["git","clone",url], check=True)
%cd {REPO}
sys.path.insert(0,".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aei_audit import indices as ix, aggregation as ag, robustness as rb
plt.rcParams.update({"font.size":9,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":.25,"figure.dpi":140})
p = pd.read_csv("data/processed/panel.csv")
INK,ACC,MUT = "#1c1c1c","#b03a2e","#cfcfcf"
print(p.shape)

## R1 — Weighting sensitivity

**Referee asks:** does "45% of inequality lost" depend on weighting by
population rather than counting states equally?

Three conventions. `population` weights by working-age people (what most
readers mean by inequality in AI use). `equal` gives Wyoming the same weight as
California, which makes it a claim about *states*, not people. `usage` describes
where conversations are rather than where people are.

In [ ]:
W = rb.weighting_sensitivity(p)
tab = W.pivot(index="wave",columns="weighting",values="pct_lost_k9").round(1)
display(tab)
print(f"range across all cells: {W.pct_lost_k9.min():.1f}% to {W.pct_lost_k9.max():.1f}%")

fig,ax=plt.subplots(figsize=(6,3.4))
for cnv in tab.columns: ax.plot(tab.index, tab[cnv], marker="o", ms=4, label=cnv)
ax.set_ylabel("% of inequality lost at k=9"); ax.legend(frameon=False,fontsize=8)
ax.set_title("The direction is robust; the magnitude is weighting-dependent",fontsize=9.5)
ax.tick_params(axis="x",labelrotation=30); plt.show()

## R2 — Leave-one-state-out

**Referee asks:** is DC or California carrying the whole result?

Shares are renormalised after each removal, which is the correct operation on
compositional data: dropping a unit does not leave the rest summing to 100.

In [ ]:
L = rb.leave_one_out(p, "2026-02")
print(f"full sample:  gini={L.attrs['full_gini']:.4f}  share_between={L.attrs['full_share_between']:.4f}")
print(f"gini range across 51 drops:           {L.gini.min():.4f} to {L.gini.max():.4f}")
print(f"share_between range across 51 drops:  {L.share_between_k9.min():.4f} to {L.share_between_k9.max():.4f}")

fig,ax=plt.subplots(figsize=(9,3.2))
s=L.sort_values("d_gini")
ax.bar(s.dropped, s.d_gini, color=[ACC if abs(v)>0.004 else MUT for v in s.d_gini])
ax.axhline(0,color=INK,lw=.8); ax.set_ylabel("Δ Gini when dropped")
ax.tick_params(axis="x",labelrotation=90,labelsize=6.5)
ax.set_title("No single state drives the Gini; watch share_between separately",fontsize=9.5)
plt.show()

## R3 — Multi-k zoning null

**Referee asks:** why nine regions? Is the upper-tail result an artifact of one
partition count?

For each k, the null is random contiguous k-partitions and the observed value is
whichever *real* government map has that k. ~2 minutes.

In [ ]:
M = rb.multi_k_zoning_null(p, "2026-02", ks=(4,6,8,9,10), n_sims=800)
display(M.dropna(subset=["real_map"]).round(3))

r=M.dropna(subset=["real_map"])
fig,ax=plt.subplots(figsize=(6.4,3.6))
ax.bar(range(len(r)), r.percentile, color=[ACC if v>90 else MUT for v in r.percentile])
ax.axhline(50,color=INK,lw=.8,ls="--"); ax.set_ylim(0,100)
ax.set_xticks(range(len(r))); ax.set_xticklabels([f"{m}\nk={k}" for m,k in zip(r.real_map,r.k)],fontsize=7.5)
ax.set_ylabel("percentile in random-map null")
ax.set_title("Every official map sits above the median; k=9 is the extreme case",fontsize=9.5)
plt.show()

## R4 — Temporal stability: order versus distribution

High Spearman rho alongside large percentage-point moves is **not** a
contradiction. Rank correlation is nearly blind to magnitude. Reporting both
separates *who is ahead* (stable) from *by how much* (not stable).

In [ ]:
T = rb.temporal_stability(p)
display(T["spearman"].round(3))
for k in T["mean_abs_rank_move"]:
    print(f"{k:22s} mean|rank move|={T['mean_abs_rank_move'][k]:5.2f}   "
          f"max|share move|={T['max_abs_share_move_pp'][k]:5.2f}pp   "
          f"top-decile persistence={T['top_decile_persistence'][k]:.2f}")

## R5 — Placebo breakpoints

**Referee asks:** the platform universe changed between 2025-11 and 2026-02, but
is that boundary actually unusual?

Characterise every wave boundary, not just the one of interest. Total variation
distance is the summed absolute share movement, halved — the share of national
usage that changed hands.

In [ ]:
B = rb.placebo_breaks(p)
display(B.round(4))

fig,ax=plt.subplots(figsize=(6.4,3.2))
cols=[ACC if "2025-11 -> 2026-02" in b else MUT for b in B.boundary]
ax.bar(range(len(B)), B.tvd_pp, color=cols)
ax.set_xticks(range(len(B))); ax.set_xticklabels(B.boundary,fontsize=7.5,rotation=20)
ax.set_ylabel("total variation distance (pp)")
ax.set_title("The platform boundary (red) is NOT the largest break",fontsize=9.5)
plt.show()

print("\nREAD THIS CAREFULLY:")
print("The platform boundary is not the biggest distributional move. The 2025-08 -> 2025-11")
print("boundary is larger, and it carries no platform change and no policy change.")
print("That does not rescue the DiD. It kills it more thoroughly: baseline wave-to-wave")
print("churn already exceeds any effect a state statute could plausibly produce.")

## Regenerate the dashboard

In [ ]:
!python scripts/build_dashboard.py
from IPython.display import IFrame
IFrame("docs/index.html", width="100%", height=620)